In [ ]:
# TOOL CALL 1: menu from a Python list
menu = [
    {"item": "Karak Chai", "price": 100},
    {"item": "Doodh Patti", "price": 120},
    {"item": "Samosa", "price": 50},
]

def get_menu():
    return menu

menu_tools = [{
    "type": "function",
    "function": {
        "name": "get_menu",
        "description": "Get the tea shop menu with prices",
        "parameters": {"type": "object", "properties": {}},   # no inputs
    },
}]

def run_with_tools(question, tools, funcs):
    chat = [
        {"role": "system", "content": "You are a tea shop assistant. Use tools to answer."},
        {"role": "user", "content": question},
    ]
    completion = llm_client.chat.completions.create(model=MODEL, messages=chat, tools=tools)
    msg = completion.choices[0].message

    if msg.tool_calls:                       # model asked for a tool
        chat.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = funcs[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args)
            chat.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
        completion = llm_client.chat.completions.create(model=MODEL, messages=chat, tools=tools)

    return completion.choices[0].message.content

print(run_with_tools("What do you have and how much?", menu_tools, {"get_menu": get_menu}))


In [ ]:
# TOOL CALL 2: external API (free weather API, no key needed)
from urllib.request import urlopen

CITIES = {"karachi": (24.86, 67.01), "lahore": (31.55, 74.34), "islamabad": (33.68, 73.05)}

def get_weather(city):
    lat, lon = CITIES.get(city.lower(), CITIES["karachi"])
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m"
    data = json.loads(urlopen(url).read())
    return {"city": city, "temperature_c": data["current"]["temperature_2m"]}

weather_tools = menu_tools + [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature of a city in Pakistan",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "e.g. Karachi"}},
            "required": ["city"],
        },
    },
}]

funcs = {"get_menu": get_menu, "get_weather": get_weather}
print(run_with_tools("I'm in Lahore. Based on the weather, what should I order from the menu?", weather_tools, funcs))


In [ ]:
def log_complaint(customer_name, phone, complaint, order_id=""):
    new_complaint = {
        "customer_name": customer_name,
        "phone": phone,
        "complaint": complaint,
        "order_id": order_id,
        "status": "Pending"
    }
    complaint_list.append(new_complaint)
    return {"success": True, "complaint_number": len(complaint_list)}


    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Save a customer complaint. Call only when you have name, phone and complaint.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_name": {"type": "string", "description": "Customer's name"},
                    "phone":         {"type": "string", "description": "Customer's phone number"},
                    "complaint":     {"type": "string", "description": "What went wrong"},
                    "order_id":      {"type": "string", "description": "Order ID, empty if not known"}
                },
                "required": ["customer_name", "phone", "complaint"]
            }
        }
    }



## Embeddings: Meaning as Numbers

**What is it?**
An embedding turns a sentence into a long list of numbers.
The numbers hold the **meaning** of the sentence.

**Why do we need it?**
Customers ask the same thing in different words:
- "Do you deliver to DHA?"
- "Can you send food to my home in DHA?"
- "Delivery hai DHA mein?"

The words are different, but the meaning is the same.
Normal search looks for the **same words**, so it fails.
Embeddings look for the **same meaning**, so they work.

**Easy example: Google Maps for sentences**
Every sentence gets a location on a map.
Sentences with the same meaning live **close together**.

| Sentence | Embedding (toy, 2 numbers) |
|---|---|
| "Do you deliver to DHA?" | [0.91, 0.12] |
| "We deliver in DHA, Gulberg and Johar Town." | [0.89, 0.15] ✅ close |
| "We are open 12 pm to 12 am." | [0.08, 0.94] ❌ far |

Real embeddings have **thousands** of numbers, not 2.

**Similarity score**
We compare two embeddings and get a score:
- close to **1** → same meaning
- close to **0** → different meaning

**Free model:** `nvidia/nemotron-3.5-embed-1b:free` on OpenRouter (same key as our chat model).

In [ ]:
# EMBEDDING 1: turn a sentence into numbers
import numpy as np

EMBED_MODEL = "nvidia/nemotron-3.5-embed-1b:free"   # free embedding model

def get_embedding(text):
    result = llm_client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(result.data[0].embedding)

vector = get_embedding("Do you deliver to DHA?")
print("How many numbers:", len(vector))
print("First 5 numbers:", vector[:5])

In [ ]:
# EMBEDDING 2: compare meanings with a similarity score
def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))   # cosine similarity

question = get_embedding("Do you deliver to DHA?")

sentences = [
    "We deliver in DHA, Gulberg and Johar Town.",
    "We are open 12 pm to 12 am.",
    "Delivery hai DHA mein?",
]

for s in sentences:
    print(round(similarity(question, get_embedding(s)), 2), "→", s)

## Embeddings + Tool Call: FAQ Search

Now we join embeddings with the tool call we learned.

**Steps:**
1. We embed all FAQs **one time** and keep the numbers.
2. Customer asks a question.
3. LLM calls our tool `search_faq`.
4. Our tool embeds the question and finds the **closest** FAQ.
5. LLM reads that FAQ and writes a friendly reply.

This idea is called **RAG** (Retrieval-Augmented Generation):
first **find** the right information, then the LLM **answers** from it.

In [ ]:
# EMBEDDING 3: FAQ search tool
faqs = [
    "We deliver in DHA, Gulberg and Johar Town. Delivery fee is Rs 150.",
    "We are open every day from 12 pm to 12 am.",
    "Chicken Biryani is spicy. Zinger Burger and Club Sandwich are not spicy.",
    "You can pay by cash, card or JazzCash.",
    "All our chicken is halal.",
]

faq_vectors = [get_embedding(f) for f in faqs]     # done only one time

def search_faq(question):
    q = get_embedding(question)
    scores = [similarity(q, v) for v in faq_vectors]
    best = int(np.argmax(scores))                   # position of the highest score
    return {"answer": faqs[best], "score": round(float(scores[best]), 2)}

print(search_faq("Kya aap JazzCash lete hain?"))

In [ ]:
# EMBEDDING 4: give search_faq to the LLM as a tool
faq_tools = [{
    "type": "function",
    "function": {
        "name": "search_faq",
        "description": "Search restaurant FAQs: delivery, timing, payment, spice, halal",
        "parameters": {
            "type": "object",
            "properties": {"question": {"type": "string", "description": "Customer's question"}},
            "required": ["question"],
        },
    },
}]

print(run_with_tools("Can you send food to my house in Gulberg?", faq_tools, {"search_faq": search_faq}))

### Quick Check
1. "Is the food halal?" and "Do you use halal chicken?" — will their embeddings be close or far?
2. Why do we embed the FAQs only **one time**?
3. Who runs `search_faq`: the LLM or our Python?

### Answer
1. **Close.** The words are different, but the meaning is the same.
2. Embedding takes time. The FAQs do not change, so we save the numbers and use them again.
3. **Our Python.** The LLM only asks for the tool. It never runs code.